In [ ]:
with push_base as (

    select
        contact_id,

        to_date(
            substr(campaign_name, 1, 8),
            'YYYYMMDD'
        ) as push_dt,

        is_sent::int as is_sent,
        is_delivered::int as is_delivered,
        is_opened::int as is_opened

    from push_act

),

subscr_min as (

    select
        contact_id,
        min(subscr_date_act::date) as first_subscr_date

    from subscr_status

    group by contact_id

),

push_client as (

    select
        pb.contact_id,

        max(pb.is_sent) as is_sent,
        max(pb.is_delivered) as is_delivered,
        max(pb.is_opened) as is_opened,

        max(
            case
                when pb.is_opened = true
                 and (
                        sm.first_subscr_date is null
                        or pb.push_dt <= sm.first_subscr_date
                     )
                then 1
                else 0
            end
        ) as opened_before_subscr

    from push_base pb

    left join subscr_min sm
        on pb.contact_id = sm.contact_id

    group by
        pb.contact_id

),

base as (

    select
        cc.client_id,
        cc.campaigns_cnt,

        coalesce(pc.is_sent, 0) as is_sent,
        coalesce(pc.is_delivered, 0) as is_delivered,
        coalesce(pc.is_opened, 0) as is_opened,

        coalesce(pc.opened_before_subscr, 0)
            as opened_before_subscr

    from client_cohorts cc

    left join push_client pc
        on cc.client_id = pc.contact_id

),

agg as (

    select
        campaigns_cnt,

        count(*) as total_clients,

        sum(is_sent) as sent_clients,
        sum(is_delivered) as delivered_clients,
        sum(is_opened) as opened_clients,

        sum(opened_before_subscr)
            as opened_before_subscr_clients

    from base

    group by campaigns_cnt

)

select
    campaigns_cnt,

    total_clients,

    sent_clients,
    delivered_clients,
    opened_clients,

    opened_before_subscr_clients,

    round(
        sent_clients * 100.0
        / nullif(total_clients, 0),
        1
    ) as sent_pct,

    round(
        delivered_clients * 100.0
        / nullif(total_clients, 0),
        1
    ) as delivered_pct,

    round(
        opened_clients * 100.0
        / nullif(total_clients, 0),
        1
    ) as opened_pct,

    round(
        opened_before_subscr_clients * 100.0
        / nullif(opened_clients, 0),
        1
    ) as opened_before_subscr_pct_from_opened,

    round(
        opened_before_subscr_clients * 100.0
        / nullif(total_clients, 0),
        1
    ) as opened_before_subscr_pct_total

from agg

order by campaigns_cnt;